In [0]:
# ----------------------------------------
# Step 1: Analyze Job Runtime (Baseline)
# ----------------------------------------

from pyspark.sql.functions import col
import time

bronze_df = spark.table("workspace.ecommerce.events_delta")

analysis_df = (
    bronze_df
        .groupBy("user_id")
        .count()
)

start_time = time.time()

result_df = analysis_df.orderBy(col("count").desc())
display(result_df.limit(10))
end_time = time.time()

print("Baseline Execution Time (seconds):",
      round(end_time - start_time, 2))

In [0]:
# Step 2: Inspect Execution Plan
analysis_df.explain("formatted")

In [0]:
analysis_df = bronze_df.groupBy("user_id").count()

In [0]:
# Step 3: Materialize Once (Avoid Recompute)
# ----------------------------------------
analysis_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.user_event_counts_temp")

In [0]:
optimized_df = spark.table("workspace.ecommerce.user_event_counts_temp")
start_time_2 = time.time()
display(optimized_df.orderBy(col("count").desc()).limit(10))
end_time_2 = time.time()

print("Optimized Execution Time (seconds):",
      round(end_time_2 - start_time_2, 2))

In [0]:
spark.sql("""
SET spark.sql.shuffle.partitions
""").show(truncate=False)